In [1]:
import pandas as pd
import numpy as np
import random
import warnings


# Suppress all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
# Backtest trades function
def backtest_trades(price_data, signal_data, tp=None, sl=None, entry_time_offset=None,
                    percentage_change=None, time_limit_minutes=None, ignore_time_interval_before=None, ignore_time_interval_after=None):
    output_data = pd.DataFrame(columns=[
        'Datetime', 'Side', 'Signal Open Price', 'Entry Price', 'TP Price', 'SL Price', 'Result', 'Duration',
        'Execution Latency', 'ROI', 'NAV', 'Ignore Reason'
    ])
    
    initial_margin = 100000
    current_margin = initial_margin
    exit_datetimes = []

    
    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']
        
        if signal_value == 0:
            continue
        elif signal_value > 0:
            side = 'Buy'
        else:
            side = 'Sell'
               
    
        adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
        signal_open_price = price_data.at[adjusted_signal_datetime, 'Open']
        
        # Ignoring signals based on open trades
        exit_datetimes.sort(key=lambda x: x[0])
        ignore_signal = False
        reason = ''
        
        if exit_datetimes:
            later_exits = [ed for ed in exit_datetimes if ed[0] > signal_datetime]


            if len(later_exits) >= 3:
                result = 'Ignored'
                reason = 'More than 2 open trades'
                ignore_signal = True
            elif len(later_exits) == 2:
                if later_exits[-1][1] == side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'Two open trades, last one with different side'
                    ignore_signal = True
            elif len(later_exits) == 1:
                if later_exits[-1][1] != side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'One open trade with the same side'
                    ignore_signal = True


        if ignore_signal:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': result,
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': reason
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue
        
        # New logic: ignore signals within a specific time interval before and after events
        if 'event_datetimes' in row and pd.notna(row['event_datetimes']):
            event_times = [pd.to_datetime(e.strip()) for e in str(row['event_datetimes']).split(',')]
            ignore_signal = any(event_datetime - pd.Timedelta(
                minutes=ignore_time_interval_before) <= signal_datetime <= event_datetime + pd.Timedelta(
                minutes=ignore_time_interval_after)
                                for event_datetime in event_times)
        if ignore_signal:
            new_row = pd.DataFrame([{
                    'Datetime': signal_datetime,
                    'Side': side,
                    'Signal Open Price': signal_open_price,
                    'Entry Price': None,
                    'TP Price': None,
                    'SL Price': None,
                    'Result': 'Ignored',
                    'Duration': '00:00:00',
                    'Execution Latency': '00:00:00',
                    'ROI': 0,
                    'NAV': current_margin,
                    'Ignore Reason': 'Signal around economic event'
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        entry_datetime, entry_price, entry_duration = determine_entry(price_data, signal_datetime, percentage_change,
                                                                      side, time_limit_minutes, entry_time_offset)
        if entry_datetime is None:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': 'Not Filled',
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': ''
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue
        
        # Update NAV on filled order
        current_margin *= (1 - 0.0002)
        
        if side == 'Buy':
            tp_price = entry_price * (1 + tp)
            sl_price = entry_price * (1 - sl)
        else:
            tp_price = entry_price * (1 - tp)
            sl_price = entry_price * (1 + sl)
        
        result, duration_str = check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side)
        exit_datetime = entry_datetime + pd.Timedelta(duration_str)

        if result in [1, -1]:
            exit_datetimes.append((exit_datetime, side))

        # Check for economic data event before the trade exit
        if 'event_datetimes' in row and pd.notna(row['event_datetimes']):
            event_times = [pd.to_datetime(e.strip()) for e in str(row['event_datetimes']).split(',')]
            for event_datetime in event_times:
                if entry_datetime < event_datetime < exit_datetime:
                    exit_datetime = event_datetime - pd.Timedelta(minutes=10)
                    if exit_datetime in price_data.index:
                        exit_price = price_data.at[exit_datetime, 'Open']
                        result = 'ended before data'
                        if (side == 'Buy' and exit_price > entry_price) or (side == 'Sell' and exit_price < entry_price):
                            result += ' with profit'
                            pct_change = (exit_price - entry_price) / entry_price if side == 'Buy' else (entry_price - exit_price) / entry_price
                            current_margin = current_margin * (1 + pct_change)
                        else:
                            result += ' with loss'
                            pct_change = (entry_price - exit_price) / entry_price if side == 'Buy' else (exit_price - entry_price) / entry_price
                            current_margin = current_margin * (1 - pct_change)
                    break

        if result not in ['ended before data with profit', 'ended before data with loss', 'ended before data with no exact price']:
            if result == 1:
                current_margin = current_margin * (1 + tp)
                current_margin *= (1 - 0.0005)
            elif result == -1:
                current_margin = current_margin * (1 - sl)
                current_margin *= (1 - 0.0005)

        
        roi = ((current_margin - initial_margin) / initial_margin) * 100
        nav = current_margin
        initial_margin = current_margin
        
        new_row = pd.DataFrame([{
            'Datetime': signal_datetime,
            'Side': side,
            'Signal Open Price': signal_open_price,
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
            'Result': result,
            'Duration': duration_str,
            'Execution Latency': format_duration(entry_duration),
            'ROI': roi,
            'NAV': nav,
            'Ignore Reason': ''
        }])
        
        output_data = pd.concat([output_data, new_row], ignore_index=True)
     
    # Ensure 'Datetime' column is in datetime format
    output_data['Datetime'] = pd.to_datetime(output_data['Datetime'])
    
        
        # Calculate Daily Return
    output_data['Date'] = output_data['Datetime'].dt.date
    daily_nav = output_data.groupby('Date')['NAV'].last().to_dict()
    daily_returns = {}
    previous_day_nav = 100000
    
    for date, nav in daily_nav.items():
        daily_return = ((nav - previous_day_nav) / previous_day_nav) * 100
        daily_returns[date] = daily_return
        previous_day_nav = nav

    output_data['Daily Return'] = output_data['Date'].map(daily_returns)
    output_data.drop(columns=['Date'], inplace=True)    
    
    return output_data


# Helper functions
def format_duration(duration):
    seconds = duration.total_seconds()
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}"

def check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side):
    result = 0
    exit_datetime = None
    subsequent_prices = price_data.loc[entry_datetime:]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy':
            if price_row['High'] >= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['Low'] <= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
        else:
            if price_row['Low'] <= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['High'] >= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
    
    if exit_datetime:
        duration = exit_datetime - entry_datetime
        duration_str = format_duration(duration)
    else:
        duration_str = '00:00:00'
        
    return result, duration_str

def determine_entry(price_data, signal_datetime, percentage_change, side, time_limit_minutes, entry_time_offset):
    adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
    if adjusted_signal_datetime not in price_data.index:
        return None, None, None
    
    adjusted_open_price = price_data.at[adjusted_signal_datetime, 'Open']
    percentage_change_price = adjusted_open_price * (1 - percentage_change) if side == 'Buy' else adjusted_open_price * (1 + percentage_change)
    
    time_limit = adjusted_signal_datetime + pd.Timedelta(minutes=time_limit_minutes)
    subsequent_prices = price_data.loc[adjusted_signal_datetime:time_limit]
    
    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy' and price_row['Low'] <= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration
        elif side == 'Sell' and price_row['High'] >= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration

    return None, None, None

In [3]:
price_data = pd.read_csv('E:\Signal Backtesting\Input\price_2023-06-01_to_2024-08-4_min.csv', parse_dates=['Datetime'],
                         index_col='Datetime')
# Re-load the signal data without setting the index
signal_data = pd.read_csv('E:\Signal Backtesting\Input\\filtered_signals.csv', parse_dates=['Datetime'])

In [4]:
# Set a random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Define the parameter ranges
tp_values = np.arange(0.009, 0.014, 0.001)
sl_values = np.arange(0.009, 0.014, 0.001)
entry_time_offset_values = np.arange(0, 120, 10)
percentage_change_values = np.arange(0.0001, 0.0015, 0.0001)
ignore_time_interval_before = np.arange(0, 1080, 60)

# Evaluation function
def evaluate(individual):
    tp, sl, entry_time_offset, percentage_change,ignore_time_interval_before = individual
    
    # Filter the data by both year and month
    month_signal_data = signal_data[(signal_data['Datetime'].dt.year == specific_year) & (signal_data['Datetime'].dt.month == specific_month)]


    # Run backtest with given parameters
    result = backtest_trades(
        price_data, month_signal_data, tp=tp, sl=sl, 
        entry_time_offset=entry_time_offset, 
        percentage_change=percentage_change, time_limit_minutes=120, ignore_time_interval_before=ignore_time_interval_before, ignore_time_interval_after=0
    )
    
    # Calculate the final NAV
    final_nav = result['NAV'].iloc[-1]
    
    # Calculate ROI
    roi = ((final_nav - 100000) / 100000) * 100
    
    return roi

# Randomly initialize an individual
def create_individual():
    return [
        np.random.choice(tp_values),
        np.random.choice(sl_values),
        np.random.choice(entry_time_offset_values),
        np.random.choice(percentage_change_values),
        np.random.choice(ignore_time_interval_before)
    ]

# Mutate an individual
def mutate(individual):
    index = random.randint(0, len(individual) - 1)
    if index == 0:
        individual[index] = np.random.choice(tp_values)
    elif index == 1:
        individual[index] = np.random.choice(sl_values)
    elif index == 2:
        individual[index] = np.random.choice(entry_time_offset_values)
    elif index == 3:
        individual[index] = np.random.choice(percentage_change_values)
    elif index == 4:
        individual[index] = np.random.choice(ignore_time_interval_before)
    return individual

# Simulated Annealing algorithm
def simulated_annealing():
    current_individual = create_individual()
    current_fitness = evaluate(current_individual)
    best_individual = list(current_individual)
    best_fitness = current_fitness
    
    initial_temperature = 1.0
    final_temperature = 0.001
    alpha = 0.99
    temperature = initial_temperature
    
    while temperature > final_temperature:
        new_individual = mutate(list(current_individual))
        new_fitness = evaluate(new_individual)
        
        if new_fitness > current_fitness or random.uniform(0, 1) < np.exp((new_fitness - current_fitness) / temperature):
            current_individual = new_individual
            current_fitness = new_fitness
        
        if current_fitness > best_fitness:
            best_individual = list(current_individual)
            best_fitness = current_fitness
        
        temperature *= alpha
    
    best_tp, best_sl, best_entry_time_offset, best_percentage_change,ignore_time_interval_before= best_individual
    optimized_roi = best_fitness
    
    print(f"Month: {specific_month}")
    print(f"Best Take Profit: {best_tp}")
    print(f"Best Stop Loss: {best_sl}")
    print(f"Best Entry Time Offset: {best_entry_time_offset}")
    print(f"Best Percentage Change: {best_percentage_change}")
    print(f"Best Ignore Time: {ignore_time_interval_before}")
    print(f"Optimized ROI: {optimized_roi:.4f}")

def optimize_for_month(year, month):
    global specific_year, specific_month
    specific_year = year
    specific_month = month
    simulated_annealing()

In [5]:
optimize_for_month(2024,6)

Month: 6
Best Take Profit: 0.011999999999999997
Best Stop Loss: 0.011999999999999997
Best Entry Time Offset: 10
Best Percentage Change: 0.00030000000000000003
Best Ignore Time: 240
Optimized ROI: 8.6017


In [6]:
optimize_for_month(2024,5)

Month: 5
Best Take Profit: 0.012999999999999996
Best Stop Loss: 0.013999999999999995
Best Entry Time Offset: 50
Best Percentage Change: 0.0012000000000000001
Best Ignore Time: 180
Optimized ROI: 8.8971


In [7]:
optimize_for_month(2024,4)

Month: 4
Best Take Profit: 0.011999999999999997
Best Stop Loss: 0.011999999999999997
Best Entry Time Offset: 10
Best Percentage Change: 0.0006000000000000001
Best Ignore Time: 1020
Optimized ROI: 17.2526


In [8]:
optimize_for_month(2024,3)

Month: 3
Best Take Profit: 0.011999999999999997
Best Stop Loss: 0.009
Best Entry Time Offset: 80
Best Percentage Change: 0.0014000000000000002
Best Ignore Time: 1020
Optimized ROI: 8.8958


In [9]:
optimize_for_month(2024,2)

Month: 2
Best Take Profit: 0.013999999999999995
Best Stop Loss: 0.009999999999999998
Best Entry Time Offset: 50
Best Percentage Change: 0.0005
Best Ignore Time: 1020
Optimized ROI: 5.5824


In [10]:
optimize_for_month(2024,1)

Month: 1
Best Take Profit: 0.013999999999999995
Best Stop Loss: 0.012999999999999996
Best Entry Time Offset: 100
Best Percentage Change: 0.0014000000000000002
Best Ignore Time: 1020
Optimized ROI: 16.8946


In [11]:
optimize_for_month(2024,7)

Month: 7
Best Take Profit: 0.009
Best Stop Loss: 0.010999999999999998
Best Entry Time Offset: 30
Best Percentage Change: 0.0005
Best Ignore Time: 840
Optimized ROI: 14.1590


In [13]:
import pandas as pd
import numpy as np
import random
from scipy.optimize import dual_annealing

# Wrapper function for backtesting with varying number of trades
def backtest_with_num_trades(price_data, signal_data, params, num_trades):
    # Filter the signal_data to only include the first num_trades signals
    filtered_signal_data = signal_data[signal_data['Signal'] != 0].iloc[:num_trades]
    if len(filtered_signal_data) < num_trades:
        return pd.DataFrame()  # Return an empty DataFrame if not enough trades
    # Call the original backtest function with the filtered signals
    result = backtest_trades(price_data, filtered_signal_data, tp=params[0], sl=params[1], 
                             entry_time_offset=params[2], 
                             percentage_change=params[3], 
                             time_limit_minutes=params[4], 
                             ignore_time_interval_before=params[5], 
                             ignore_time_interval_after=params[6])
    return result

# Evaluation function for the optimizer
def evaluate(params, price_data, signal_data, num_trades):
    # Run backtest with initial number of trades
    initial_result = backtest_with_num_trades(price_data, signal_data, params, num_trades)
    if initial_result.empty or 'NAV' not in initial_result.columns or initial_result['NAV'].empty:
        print(f"Insufficient data for initial evaluation with num_trades={num_trades}")
        return float('inf')  # Return a high value to indicate poor performance if not enough data
    final_nav_initial = initial_result['NAV'].iloc[-1]
    initial_roi = ((final_nav_initial - 100000) / 100000) * 100

    # Use the same parameters to backtest with a different set of trades
    eval_result = backtest_with_num_trades(price_data, signal_data, params, num_trades)
    if eval_result.empty or 'NAV' not in eval_result.columns or eval_result['NAV'].empty:
        print(f"Insufficient data for evaluation with num_trades={num_trades}")
        return float('inf')  # Return a high value to indicate poor performance if not enough data
    final_nav_eval = eval_result['NAV'].iloc[-1]
    eval_roi = ((final_nav_eval - 100000) / 100000) * 100

    # The objective is to maximize the average ROI of both tests
    combined_roi = (initial_roi + eval_roi) / 2
    return -combined_roi  # Minimize the negative ROI

# Function to convert index to the actual parameter values
def convert_to_actual_params(indices):
    indices = np.round(indices).astype(int)  # Ensure indices are integers
    tp = tp_values[indices[0]]
    sl = sl_values[indices[1]]
    entry_time_offset = entry_time_offset_values[indices[2]]
    percentage_change = percentage_change_values[indices[3]]
    ignore_time_interval_before = ignore_time_interval_before_values[indices[4]]
    time_limit_minutes = 120  # Example value, can be adjusted
    ignore_time_interval_after = 0  # Example value, can be adjusted
    return tp, sl, entry_time_offset, percentage_change, time_limit_minutes, ignore_time_interval_before, ignore_time_interval_after

# Simulated Annealing optimization
def optimize_parameters(price_data, signal_data, num_trades, max_iterations):
    # Define the parameter ranges
    global tp_values, sl_values, entry_time_offset_values, percentage_change_values, ignore_time_interval_before_values
    tp_values = np.arange(0.009, 0.014, 0.001)
    sl_values = np.arange(0.009, 0.014, 0.001)
    entry_time_offset_values = np.arange(0, 120, 10)
    percentage_change_values = np.arange(0.0001, 0.0015, 0.0001)
    ignore_time_interval_before_values = np.arange(0, 1080, 60)

    bounds = [(0, len(tp_values) - 1),  # tp
              (0, len(sl_values) - 1),  # sl
              (0, len(entry_time_offset_values) - 1),  # entry_time_offset
              (0, len(percentage_change_values) - 1),  # percentage_change
              (0, len(ignore_time_interval_before_values) - 1)]  # ignore_time_interval_before

    def bounded_evaluate(indices, price_data, signal_data, num_trades):
        params = convert_to_actual_params(indices)
        result = evaluate(params, price_data, signal_data, num_trades)
        if np.isnan(result) or np.isinf(result):
            print(f"Invalid evaluation result with params={params}, result={result}")
            return float('inf')  # Return a high value to indicate poor performance if the result is invalid
        return result

    result = dual_annealing(bounded_evaluate, bounds, args=(price_data, signal_data, num_trades), maxiter=max_iterations)
    optimized_indices = np.round(result.x).astype(int)
    optimized_params = convert_to_actual_params(optimized_indices)
    return optimized_params, -result.fun

# Function to filter data for a specific month and year
def filter_data_for_month_year(signal_data, year, month):
    return signal_data[(signal_data['Datetime'].dt.year == year) & (signal_data['Datetime'].dt.month == month)]

# Function to split data sequentially
def split_data_sequentially(signal_data, initial_num_trades, eval_num_trades):
    initial_set = signal_data.iloc[:initial_num_trades]
    eval_set = signal_data.iloc[initial_num_trades:initial_num_trades + eval_num_trades]
    return initial_set, eval_set

# Function to find the optimal number of trades
def find_optimal_number_of_trades(price_data, signal_data, max_trades, max_iterations):
    best_num_trades = 0
    best_combined_roi = float('-inf')
    best_params = None

    for num_trades in range(1, max_trades + 1):
        initial_set, eval_set = split_data_sequentially(signal_data, num_trades, num_trades)
        if len(initial_set) < num_trades or len(eval_set) < num_trades:
            print(f"Skipping num_trades={num_trades} due to insufficient data")
            continue  # Skip if not enough trades
        optimized_params, combined_roi = optimize_parameters(price_data, initial_set, num_trades, max_iterations)
        
        # Re-evaluate with evaluation set
        eval_result = backtest_with_num_trades(price_data, eval_set, optimized_params, num_trades)
        if eval_result.empty or 'NAV' not in eval_result.columns or eval_result['NAV'].empty:
            print(f"Skipping evaluation for num_trades={num_trades} due to insufficient data")
            continue  # Skip if not enough data
        final_nav_eval = eval_result['NAV'].iloc[-1]
        eval_roi = ((final_nav_eval - 100000) / 100000) * 100

        print(f"Num Trades: {num_trades}, Combined ROI: {combined_roi:.2f}%, Eval ROI: {eval_roi:.2f}%")

        if combined_roi > best_combined_roi:
            best_combined_roi = combined_roi
            best_num_trades = num_trades
            best_params = optimized_params

    return best_num_trades, best_params, best_combined_roi

# Example usage
def main():

    # Filter signal data for a specific month and year
    # specific_year = 2024
    # specific_month = 7
    # filtered_signal_data = filter_data_for_month_year(signal_data, specific_year, specific_month)

    max_trades = 10
    max_iterations = 50

    best_num_trades, best_params, best_combined_roi = find_optimal_number_of_trades(price_data, signal_data, max_trades, max_iterations)
    print(f"Best Number of Trades: {best_num_trades}")
    print(f"Best Optimized Parameters: {best_params}")
    print(f"Best Combined ROI: {best_combined_roi:.2f}%")

    # Final backtest with optimized parameters and best number of trades
    final_output_data = backtest_with_num_trades(price_data, signal_data, best_params, best_num_trades)
    print(final_output_data)

if __name__ == "__main__":
    main()


Insufficient data for initial evaluation with num_trades=1
Invalid evaluation result with params=(0.012999999999999996, 0.012999999999999996, 0, 0.0012000000000000001, 120, 720, 0), result=inf
Insufficient data for initial evaluation with num_trades=1
Invalid evaluation result with params=(0.009999999999999998, 0.009999999999999998, 80, 0.0007000000000000001, 120, 960, 0), result=inf
Insufficient data for initial evaluation with num_trades=1
Invalid evaluation result with params=(0.012999999999999996, 0.009, 100, 0.0007000000000000001, 120, 120, 0), result=inf
Insufficient data for initial evaluation with num_trades=1
Invalid evaluation result with params=(0.011999999999999997, 0.011999999999999997, 50, 0.0013000000000000002, 120, 300, 0), result=inf
Insufficient data for initial evaluation with num_trades=1
Invalid evaluation result with params=(0.012999999999999996, 0.009, 90, 0.0001, 120, 120, 0), result=inf
Insufficient data for initial evaluation with num_trades=1
Invalid evaluati

ValueError: Stopping algorithm because function create NaN or (+/-) infinity values even with trying new random parameters